# Loading an isolated sign language dataset

This notebook shows how to load an **isolated-sign** dataset (e.g. LSFB-ISOL) using a config preset (`LSFBIsolConfig`).

In isolated mode (`isolated=True`), each sample is a single, pre-segmented sign clip with a `label`/`label_id` pair, instead of the multi-frame recordings with temporal annotations used in continuous mode.

In [1]:
import random

from sldl import SignLanguageDataset
from sldl.configs import LSFBIsolConfig

from sign_language_tools.player import VideoPlayer
from sign_language_tools.pose.mediapipe.edges import UPPER_POSE_EDGES, HAND_EDGES, LIPS_EDGES, EYE_EDGES

## Build the dataset

`LSFBIsolConfig` derives `shards_url` and video paths from `root`, `variant` (vocabulary size: `"500"`, `"750"`, `"2000"` or `"all"`) and `split`. The commented-out block below shows the equivalent manual `SignLanguageDataset(...)` call, for reference — everything it sets is inferred automatically by the config.

In [2]:
root = "F:/datasets/sign-language/lsfb-isol"

dataset = SignLanguageDataset.from_config(LSFBIsolConfig(
    root=root,
    variant='500',
    split='testing',
    load_videos=True,
))
len(dataset)

Loading dataset [file:F:/datasets/sign-language/lsfb-isol/shards/500/shard_000000.tar]...


Loading samples: 9692 samples [00:11, 843.68 samples/s]

Loaded 9692 samples.


9692

## Inspect a sample

Isolated samples expose `label` (the sign string) and `label_id` (its class index) in addition to the usual `id`, `signer_id`, `poses` and `n_frames` fields. Since `load_videos=True` here, a decoded `video` tensor is also included.

In [3]:
example_sample = dataset[0]
print("Sample ID:", example_sample['id'])
print("Label:", example_sample['label'])
print("Signer:", example_sample['signer_id'])
print("Nb. of frames:", example_sample['n_frames'])
print("Pose sequence - body parts:", set(example_sample['poses'].keys()))
print("Pose sequence - upper pose shape:", example_sample['poses']['upper_pose'].shape)
print("Video shape:", example_sample['video'].shape)

Sample ID: CLSFBI0301A_S008_B_12100_12437
Label: bonjour
Signer: S008
Nb. of frames: 18
Pose sequence - body parts: {'right_eye', 'left_eye', 'right_hand', 'left_hand', 'lips', 'upper_pose'}
Pose sequence - upper pose shape: (18, 23, 3)
Video shape: torch.Size([18, 3, 480, 600])


## Sample and visualize signs for a given label

`dataset.samples` holds the raw (un-transformed) sample dicts, so we can filter by `label` without triggering pose/video transforms, then fetch the full samples (with video + poses) only for the ones we want to inspect.

In [4]:
def get_sample_per_label(label: str, max_n=10) -> list[dict]:
    sample_indices = [i for i, s in enumerate(dataset.samples) if s['label'] == label]
    random.shuffle(sample_indices)
    return [dataset[i] for i in sample_indices[:max_n]]

In [5]:
def inspect_sample(sample):
    player = VideoPlayer()
    player.attach_video_tensor(sample['video'].permute(0, 2, 3, 1), fps=50, name='video')
    # player.attach_empty(800, 600, name='skeleton')
    player.attach_poses(sample['poses']['upper_pose'], UPPER_POSE_EDGES, parent_name='video')
    player.attach_poses(sample['poses']['left_hand'], HAND_EDGES, edge_color=(255, 0, 0), parent_name='video')
    player.attach_poses(sample['poses']['right_hand'], HAND_EDGES, edge_color=(0, 0, 255), parent_name='video')
    player.attach_poses(sample['poses']['lips'], LIPS_EDGES, edge_color=(0, 0, 255), parent_name='video')
    player.attach_poses(sample['poses']['left_eye'], EYE_EDGES, edge_color=(255, 255, 255), parent_name='video')
    player.attach_poses(sample['poses']['right_eye'], EYE_EDGES, edge_color=(255, 255, 255), parent_name='video')
    player.play(speed=0.2)

In [6]:
for sample in get_sample_per_label('rencontrer', max_n=10):
    inspect_sample(sample)

# Use a Collator with Data Loaders

In [9]:
from torch.utils.data import DataLoader

from sldl.collator import SignLanguageCollator

collator = SignLanguageCollator()
data_loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=0, collate_fn=collator)

batch = next(iter(data_loader))
print("Batch:", batch.keys())
print("Poses:", batch['poses']['upper_pose'].shape)
print("Videos:", batch['video'].shape)
print("Masks:", batch['masks'].shape)

Batch: dict_keys(['id', 'signer_id', 'poses', 'label_id', 'label', 'n_frames', '__key__', 'video', 'masks', 'lengths'])
Poses: torch.Size([4, 42, 23, 3])
Videos: torch.Size([4, 42, 3, 480, 600])
Masks: torch.Size([4, 42])
